# 2. Hyperparameter tuning
Sweeps on the tuning DAGs (101–105), their results, and the candidate check that picks the config
for the main experiments in `3_main_experiments.ipynb`. Shared settings come from `1_mle.ipynb` (its `config` cell); this
notebook's own tuning and selection settings are tagged `config` too, so `3_main_experiments.ipynb` can load them.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import logging

import numpy as np
import pandas as pd
import torch.nn as nn

from ndbn.experiments import run_sweep, run_configs
import ndbn.plotting as plotting
from ndbn.notebook_config import load_notebook_config

logging.getLogger("pgmpy").setLevel(logging.ERROR)

# shared settings (RESULTS_DIR, CARD, SAMPLE_SIZES, speed settings, ...) from 1_mle.ipynb
globals().update(load_notebook_config("1_mle.ipynb"))
print("results folder:", RESULTS_DIR)

results folder: results5-parallel


In [2]:
from IPython.display import HTML, display
display(HTML("""<style>
.cell-output-ipywidget-background { background-color: transparent !important; }
:root {
  --jp-widgets-color: var(--vscode-editor-foreground);
  --jp-widgets-font-size: var(--vscode-editor-font-size);
}
</style>"""))

## Sweeps
Fixed graph size, varying only the estimator. Writes its own `sweep_tune.csv`, so it
cannot contaminate the main grid's `results.csv`. All tuning runs on separate DAGs
(101–105, `TUNE_DAG_OFFSET`); the main experiment's DAGs 1–5 are kept for final evaluation.

In [3]:
%pip install wandb   
# Offline alternative (sync later with `wandb sync`):  os.environ["WANDB_MODE"] = "offline"

Note: you may need to restart the kernel to use updated packages.


In [4]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Currently logged in as: kay-asp (kay-asp-wits-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import os
import torch.nn as nn

# All tuning (architecture sweep, no-ES diagnostic, hyperparameter sweep) runs on
# separate DAGs 101..100+SWEEP_DAGS from the same generator. The main experiment's
# DAGs (1..SIM) are kept for final evaluation only.
TUNE_DAG_OFFSET = 100
assert TUNE_DAG_OFFSET >= SIM, "tuning DAGs overlap the evaluation DAGs"

WANDB_PROJECT = "neural-cpds-tune"

SWEEP_CSV     = os.path.join(RESULTS_DIR, "sweep_tune.csv")
HIST_DIR      = os.path.join(RESULTS_DIR, "histories_tune")

SWEEP_N       = 20
SWEEP_ALPHA   = 0.5
SWEEP_DAGS    = 5
SWEEP_K       = [2, 6, 10]          # widen to [2, 4, 6, 8, 10] if time allows
SWEEP_SAMPLES = [600, 1400]

ARMS = [
    {"name": "nn_linear",     "hidden_dims": (),        "activation": None},
    {"name": "nn_32",         "hidden_dims": (32,),     "activation": nn.ReLU},
    {"name": "nn_64_32",      "hidden_dims": (64, 32),  "activation": nn.ReLU},
    {"name": "nn_128_64",     "hidden_dims": (128, 64), "activation": nn.ReLU},
    {"name": "nn_32_nowd",    "hidden_dims": (32,), "activation": nn.ReLU,
                              "weight_decay": 0.0},
    {"name": "nn_32_lr3e3",   "hidden_dims": (32,), "activation": nn.ReLU,
                              "lr": 3e-3},
    {"name": "nn_32_long",    "hidden_dims": (32,), "activation": nn.ReLU,
                              "n_epochs": 1000, "patience": 50},
]

In [6]:
sweep_df = run_sweep(
    arms=ARMS, k_values=SWEEP_K, n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=SWEEP_SAMPLES, card=CARD,
    val_frac=VAL_FRAC, n_epochs=N_EPOCHS,
    wandb_project=WANDB_PROJECT, sweep_csv=SWEEP_CSV, hist_dir=HIST_DIR,
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES),   # main experiment's sample budget
    n_test=N_TEST,                                         # true KL on cached test.csv
    dag_offset=TUNE_DAG_OFFSET,                            # tuning DAGs, not evaluation
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

sweep runs:   0%|          | 0/210 [00:00<?, ?it/s]

KeyboardInterrupt: 

### No-early-stopping diagnostic
Patience > number of epochs, so nothing stops training. Shows how fast and how far each
architecture overfits at k = 10, N = 600 — compare `train_loss`, `val_loss` and `gap`
per epoch in W&B. Own project, CSV and history folder, so it can't mix with the main sweep.

In [ ]:
NO_ES_EPOCHS = 500
NO_ES_ARMS = [{**arm, "n_epochs": NO_ES_EPOCHS, "patience": 1000}
              for arm in ARMS if arm["name"] in ("nn_32", "nn_64_32", "nn_128_64")]

no_es_df = run_sweep(
    arms=NO_ES_ARMS, k_values=[10], n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=[600], card=CARD,
    val_frac=VAL_FRAC, n_epochs=NO_ES_EPOCHS,
    wandb_project="neural-cpds-no-early-stopping-tune",
    sweep_csv=os.path.join(RESULTS_DIR, "sweep_no_es_tune.csv"),
    hist_dir=os.path.join(RESULTS_DIR, "histories_no_es_tune"),
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES),   # main experiment's sample budget
    n_test=N_TEST,                                         # true KL on cached test.csv
    dag_offset=TUNE_DAG_OFFSET,                            # tuning DAGs, not evaluation
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

### Hyperparameter sweep (W&B, Bayesian)
Tuning runs on **separate DAGs** (`dag_idx` > `TUNE_DAG_OFFSET`) drawn from the same generator,
never on the main experiment's DAGs. The chosen setting is then frozen and evaluated on the
main experiment's DAGs, so their test sets never influence any choice.

Each trial is one config scored on every tuning DAG × sample size for the given k. Because
the tuning DAGs are disjoint from evaluation, selecting on `mean_kl` (true KL on the tuning
DAGs' `test.csv`) is legitimate; `mean_val_loss` (validation-only) is the alternative. Both
are logged. AdamW, so `weight_decay` is decoupled and actually regularises. Start with
k = 10, then check how the frozen setting transfers to k = 2, 6.

In [7]:
from functools import partial
from ndbn.experiments import sweep_trial

HP_PROJECT      = "neural-cpds-hparams"
HP_K            = 10
HP_TRIALS       = 50
HP_METRIC       = "mean_kl"      # or "mean_val_loss" (validation-only selection)

HP_CONFIG = {
    "name": f"{HP_METRIC}_c{CARD}_k{HP_K}",                 # sweep name shown in W&B (else a random id)
    "description": f"card {CARD}, in-degree {HP_K}, tuning DAGs {TUNE_DAG_OFFSET + 1}-{TUNE_DAG_OFFSET + SWEEP_DAGS}, "
                   f"N = {SWEEP_SAMPLES}, minimising {HP_METRIC}",
    "method": "bayes",
    "metric": {"name": HP_METRIC, "goal": "minimize"},
    "parameters": {
        "hidden_dims":  {"values": ["32", "64_32", "128_64", "256_128"]},
        "lr":           {"distribution": "log_uniform_values", "min": 3e-4, "max": 1e-2},
        "weight_decay": {"distribution": "log_uniform_values", "min": 1e-5, "max": 1e-1},
        "dropout":      {"values": [0.0, 0.1, 0.2, 0.3, 0.5]},
        "patience":     {"values": [20, 50, 100]},
        "batch_size":   {"values": [32, 64, 128]},
        "optimizer":    {"value": "adamw"},
        # "activation": {"values": ["relu", "tanh", "gelu"]},   # optional; default relu
        "n_epochs":     {"value": 1000},
    },
}

In [ ]:
HP_K = 10
hp_sweep_id = 'p8lmbfxj' # wandb.sweep(HP_CONFIG, project=HP_PROJECT)   # or reuse an existing id to resume

run_trial = partial(
    sweep_trial, k_values=[HP_K], n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=SWEEP_SAMPLES, card=CARD, val_frac=VAL_FRAC,
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES), n_test=N_TEST,
    # one file for every sweep: each row records all its settings + sweep_id
    trial_csv=TRIALS_CSV,
    dag_offset=TUNE_DAG_OFFSET,
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

from tqdm.auto import tqdm
bar = tqdm(total=HP_TRIALS, desc="sweep trials")

def trial():
    try:
        run_trial()
    finally:
        bar.update()          # counts even if a trial fails

wandb.agent(hp_sweep_id, project=HP_PROJECT, count=HP_TRIALS, function=trial)
bar.close()

Create sweep with ID: p8lmbfxj
Sweep URL: https://wandb.ai/kay-asp-wits-university/neural-cpds-hparams/sweeps/p8lmbfxj


sweep trials:   0%|          | 0/50 [00:00<?, ?it/s]

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: ckm5nlz5 with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0
wandb: 	hidden_dims: 256_128
wandb: 	lr: 0.0006490571148348818
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 50
wandb: 	weight_decay: 0.02142725152145705


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,1.88892
mean_val_loss,9.7178
sample_size,1400
std_kl,0.20729


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: 6bd3krb1 with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0
wandb: 	hidden_dims: 32
wandb: 	lr: 0.0010794742484677405
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 100
wandb: 	weight_decay: 0.018281353070021347


...

mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,1.86257
mean_val_loss,9.71204
sample_size,1400
std_kl,0.22106


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: b46fhhhc with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.5
wandb: 	hidden_dims: 256_128
wandb: 	lr: 0.0006352461576126954
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 20
wandb: 	weight_decay: 0.05778575724018419


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,1.84818
mean_val_loss,9.70185
sample_size,1400
std_kl,0.21045


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: c8cpbwfb with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.5
wandb: 	hidden_dims: 256_128
wandb: 	lr: 0.0007442032736754118
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 20
wandb: 	weight_decay: 0.02929431025378846


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,1.84665
mean_val_loss,9.69104
sample_size,1400
std_kl,0.22756


In [ ]:
from functools import partial
from ndbn.experiments import sweep_trial

HP_PROJECT      = "neural-cpds-hparams"
HP_K            = 6
HP_TRIALS       = 50
HP_METRIC       = "mean_kl"      # or "mean_val_loss" (validation-only selection)

HP_CONFIG = {
    "name": f"{HP_METRIC}_c{CARD}_k{HP_K}",                 # sweep name shown in W&B (else a random id)
    "description": f"card {CARD}, in-degree {HP_K}, tuning DAGs {TUNE_DAG_OFFSET + 1}-{TUNE_DAG_OFFSET + SWEEP_DAGS}, "
                   f"N = {SWEEP_SAMPLES}, minimising {HP_METRIC}",
    "method": "bayes",
    "metric": {"name": HP_METRIC, "goal": "minimize"},
    "parameters": {
        "hidden_dims":  {"values": ["32", "64_32", "128_64", "256_128"]},
        "lr":           {"distribution": "log_uniform_values", "min": 3e-5, "max": 1e-3},
        "weight_decay": {"value": 1e-4},
        "dropout":      {"values": [0.0, 0.1]},
        "patience":     {"values": [5, 10, 20]},
        "batch_size":   {"values": [64, 128, 256]},
        "optimizer":    {"value": "adamw"},
        # "activation": {"values": ["relu", "tanh", "gelu"]},   # optional; default relu
        "n_epochs":     {"value": 1000},
    },
}

In [ ]:
HP_K = 6
hp_sweep_id = 'ey794fwc'  # or reuse an existing id to resume

run_trial = partial(
    sweep_trial, k_values=[HP_K], n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=SWEEP_SAMPLES, card=CARD, val_frac=VAL_FRAC,
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES), n_test=N_TEST,
    # one file for every sweep: each row records all its settings + sweep_id
    trial_csv=TRIALS_CSV,
    dag_offset=TUNE_DAG_OFFSET,
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

from tqdm.auto import tqdm
bar = tqdm(total=HP_TRIALS, desc="sweep trials")

def trial():
    try:
        run_trial()
    finally:
        bar.update()          # counts even if a trial fails

wandb.agent(hp_sweep_id, project=HP_PROJECT, count=HP_TRIALS, function=trial)
bar.close()

Create sweep with ID: ey794fwc
Sweep URL: https://wandb.ai/kay-asp-wits-university/neural-cpds-hparams/sweeps/ey794fwc


sweep trials:   0%|          | 0/50 [00:00<?, ?it/s]

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: lkee28ae with config:
wandb: 	batch_size: 256
wandb: 	dropout: 0.1
wandb: 	hidden_dims: 256_128
wandb: 	lr: 7.667716223070097e-05
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 5
wandb: 	weight_decay: 0.0001


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,0.9113
mean_val_loss,8.38638
sample_size,1400
std_kl,0.16738


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: lb30h9qy with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.1
wandb: 	hidden_dims: 64_32
wandb: 	lr: 0.0007043242037296014
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 20
wandb: 	weight_decay: 0.0001


...

mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,0.59099
mean_val_loss,7.97251
sample_size,1400
std_kl,0.07795


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: 2bx6snxj with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.1
wandb: 	hidden_dims: 32
wandb: 	lr: 9.011172768233074e-05
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 20
wandb: 	weight_decay: 0.0001


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0.465
mean_kl,1.07894
mean_val_loss,8.60762
sample_size,1400
std_kl,0.23531


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: b3t1w1t7 with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0
wandb: 	hidden_dims: 32
wandb: 	lr: 7.364868362695847e-05
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 10
wandb: 	weight_decay: 0.0001


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0.76
mean_kl,1.44201
mean_val_loss,8.99725
sample_size,1400
std_kl,0.31428


In [ ]:
from functools import partial
from ndbn.experiments import sweep_trial

HP_PROJECT      = "neural-cpds-hparams"
HP_K            = 10
HP_TRIALS       = 30
HP_METRIC       = "mean_kl"      # or "mean_val_loss" (validation-only selection)

HP_CONFIG = {
    "name": f"{HP_METRIC}_c{CARD}_k{HP_K}",                 # sweep name shown in W&B (else a random id)
    "description": f"card {CARD}, in-degree {HP_K}, tuning DAGs {TUNE_DAG_OFFSET + 1}-{TUNE_DAG_OFFSET + SWEEP_DAGS}, "
                   f"N = {SWEEP_SAMPLES}, minimising {HP_METRIC}",
    "method": "bayes",
    "metric": {"name": HP_METRIC, "goal": "minimize"},
    "parameters": {
        "hidden_dims":  {"values": ["32", "64_32", "128_64", "256_128"]},
        "lr":           {"distribution": "log_uniform_values", "min": 1e-5, "max": 1e-3},
        "weight_decay": {"distribution": "log_uniform_values", "min": 1e-5, "max": 1e-1},
        "dropout":      {"values": [0.0, 0.1, 0.3, 0.5]},
        "patience":     {"values": [20, 50, 100]},
        "batch_size":   {"values": [32, 64, 128]},
        "optimizer":    {"value": "adamw"},
        # "activation": {"values": ["relu", "tanh", "gelu"]},   # optional; default relu
        "n_epochs":     {"value": 1000},
    },
}

In [ ]:
HP_K = 2
hp_sweep_id = "gqtytopa" #wandb.sweep(HP_CONFIG, project=HP_PROJECT)

run_trial = partial(
    sweep_trial, k_values=[HP_K], n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=SWEEP_SAMPLES, card=CARD, val_frac=VAL_FRAC,
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES), n_test=N_TEST,
    # one file for every sweep: each row records all its settings + sweep_id
    trial_csv=TRIALS_CSV,
    dag_offset=TUNE_DAG_OFFSET,
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

from tqdm.auto import tqdm
bar = tqdm(total=HP_TRIALS, desc="sweep trials")

def trial():
    try:
        run_trial()
    finally:
        bar.update()          # counts even if a trial fails

wandb.agent(hp_sweep_id, project=HP_PROJECT, count=HP_TRIALS, function=trial)
bar.close()

Create sweep with ID: gqtytopa
Sweep URL: https://wandb.ai/kay-asp-wits-university/neural-cpds-hparams/sweeps/gqtytopa


sweep trials:   0%|          | 0/50 [00:00<?, ?it/s]

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\Kaylin\_netrc.
wandb: Agent Starting Run: 4wbugbg0 with config:
wandb: 	batch_size: 32
wandb: 	dropout: 0.1
wandb: 	hidden_dims: 128_64
wandb: 	lr: 0.007141661176827854
wandb: 	n_epochs: 1000
wandb: 	optimizer: adamw
wandb: 	patience: 20
wandb: 	weight_decay: 0.00108400144426591


## Hyperparameter sweep results
From `hparam_trials.csv` (every sweep; each row records all its settings and its `sweep_id`),
filtered with `TRIAL_FILTERS` (any recorded column). Weight decay was found to have a negligible effect
on KL, so it is fixed at the AdamW default of 0.01 (`FIXED`, set in the selection cell above); trials whose
settings are then identical (after rounding, or `SNAP`) are averaged automatically. One table per (in-degree, sample size): the configs
whose mean KL over the tuning DAGs is within one `kl_se` of that pair's best. Then one table of the
settings each pair's top configs share, and the configs that are in the top of more than one table, matched on `match_on` (add e.g. `"lr"`, rounded to 1
significant figure).

In [ ]:
# Sweep-result selection (used by the sweep results and candidate cells below)
TRIAL_FILTERS = dict(card=CARD)   # any recorded column, e.g. alpha=SWEEP_ALPHA, sweep_id=...
N_SE          = 2                 # a config is "top" if its mean KL is within N_SE kl_se of the best

# Weight decay was found to have a negligible effect on KL (a 10x change in weight decay moves
# regret by <= ~0.2 percentage points with the other settings fixed), so it is fixed at the AdamW
# default 0.01: trials that then have identical settings are averaged automatically, and every
# chosen config runs with weight decay 0.01.
FIXED         = {"weight_decay": 0.01}
SNAP          = (("lr", (3,6)), ("dropout", (1,3)) )                # e.g. (("lr", (1, 3)),) to snap lr to 1e-4, 3e-4, 1e-3, ...

In [ ]:
# hparam_trials.csv (all sweeps) plus any older per-k files
trials = plotting.load_trials(RESULTS_DIR)

# TRIAL_FILTERS, N_SE, FIXED and SNAP come from the selection cell above (weight decay fixed: negligible
# effect; trials that end up identical are averaged). Override here to explore, e.g. N_SE = 2.
tables = plotting.summarise_trials(trials, n_se=N_SE, fixed=FIXED, snap=SNAP, **TRIAL_FILTERS)

# per table: the settings all its top configs share (read down a column to compare tables)
shared  = plotting.shared_settings_table(tables)

# configs in the top of more than one table; match_on=(..., "lr") also matches lr (rounded)
common  = plotting.common_configs(tables, match_on=("hidden_dims", "dropout", "patience", "batch_size"))

# full settings of every top config
# lr rounded to 1 significant figure; weight decay fixed at the AdamW default for the runs
top_values = plotting.top_config_values(tables)   # uses the same FIXED / SNAP

in-degree 6, N = 600: best 0.7806 ± 0.0056 -> 6 of 55 configs (weight_decay fixed at 0.01) (lr snapped to 3-6 grid) (dropout snapped to 1-3 grid) within 2 kl_se


,run_name,mean_kl,kl_se,mean_val_loss,n_trials,hidden_dims,lr,dropout,patience,batch_size,stored_params,best_epoch_mean,fit_time_s,sweep_id
0,40_256x128_lr3e-4,0.7806,0.0056,8.0983,1,"[256, 128]",0.00030,0.1,20,256,719144.0,135.7500,47.9705,ey794fwc
1,k6_256_128_25,0.7841,0.0057,8.0653,4,"[256, 128]",0.00030,0.1,20,128,719144.0,NaN,40.7341,NaN
2,k6_256_128_37,0.7887,0.0058,8.0391,2,"[256, 128]",0.00030,0.3,20,32,719144.0,NaN,66.2205,NaN
3,35_256x128_lr2e-4,0.7899,0.0056,8.0562,4,"[256, 128]",0.00030,0.1,20,64,719144.0,78.3867,52.9790,ey794fwc
4,k6_256_128_39,0.7914,0.0059,8.0556,5,"[256, 128]",0.00030,0.0,20,128,719144.0,NaN,32.2723,NaN
5,28_256x128_lr8e-5,0.7917,0.0053,8.1431,1,"[256, 128]",0.00006,0.1,20,64,719144.0,189.2400,115.3974,ey794fwc


in-degree 6, N = 1400: best 0.3684 ± 0.0035 -> 2 of 55 configs (weight_decay fixed at 0.01) (lr snapped to 3-6 grid) (dropout snapped to 1-3 grid) within 2 kl_se


,run_name,mean_kl,kl_se,mean_val_loss,n_trials,hidden_dims,lr,dropout,patience,batch_size,stored_params,best_epoch_mean,fit_time_s,sweep_id
0,k6_128_64_9,0.3684,0.0035,7.8785,1,"[128, 64]",0.0003,0.1,50,128,195752.0,NaN,74.9822,NaN
1,k6_128_64_5,0.3716,0.0037,7.8677,1,"[128, 64]",0.0003,0.1,100,128,195752.0,NaN,111.8904,NaN


in-degree 10, N = 600: best 2.2461 ± 0.0085 -> 4 of 25 configs (weight_decay fixed at 0.01) (lr snapped to 3-6 grid) (dropout snapped to 1-3 grid) within 2 kl_se


...

,in_degree,sample_size,n_top,hidden_dims,lr,dropout,patience,batch_size
0,6,600,6,"[256, 128]",–,–,20,–
1,6,1400,2,"[128, 64]",≈3e-4,0.1,–,128
2,10,600,4,"[256, 128]",≈3e-4,0.0,–,–
3,10,1400,3,"[256, 128]",–,0.3,–,–


configs (matched on hidden_dims, dropout, patience, batch_size) in the top (within 2 kl_se) of >= 2 of 4 tables: 2


,hidden_dims,dropout,patience,batch_size,n_tables,top_in,n_trials
0,"[256, 128]",0.0,20,128,2,"k6 N600, k10 N600",2
1,"[256, 128]",0.3,20,32,2,"k6 N600, k10 N1400",2


top configs (within 2 kl_se), lr snapped to 3-6 grid; dropout snapped to 1-3 grid; weight_decay fixed at 0.01 for runs, identical configs merged:


,run_name,in_tables,n_tables,n_trials,hidden_dims,activation,lr,dropout,patience,batch_size
0,07_256x128_lr3e-4,"k6 N600, k10 N600",2,11,"[256, 128]",ReLU,0.00030,0.0,20,128
1,09_256x128_lr3e-4,k10 N600,1,6,"[256, 128]",ReLU,0.00030,0.0,20,32
2,35_256x128_lr2e-4,k6 N600,1,4,"[256, 128]",ReLU,0.00030,0.1,20,64
3,k6_256_128_25,k6 N600,1,4,"[256, 128]",ReLU,0.00030,0.1,20,128
4,10_256x128_lr3e-4,k10 N600,1,3,"[256, 128]",ReLU,0.00030,0.0,20,64
5,25_256x128_lr3e-4,k10 N600,1,3,"[256, 128]",ReLU,0.00030,0.0,100,32
6,40_256x128_lr3e-4,k10 N1400,1,3,"[256, 128]",ReLU,0.00030,0.3,50,32
7,49_256x128_lr6e-4,k10 N1400,1,3,"[256, 128]",ReLU,0.00060,0.3,20,64
8,k6_256_128_37,k6 N600,1,2,"[256, 128]",ReLU,0.00030,0.3,20,32
9,28_256x128_lr8e-5,k6 N600,1,1,"[256, 128]",ReLU,0.00006,0.1,20,64


## Candidates
### Run chosen configs on all (in-degree, sample size)
Runs specific configs on the tuning DAGs (101–105) for the in-degrees in `CAND_SWEEPS` (k = 6, 10)
and every sample size in `SWEEP_SAMPLES`, so they are compared like for like (same DAGs, data and seeds). Evaluation DAGs stay untouched. `CANDIDATES` picks rows of the
top-config table above (`top_values.loc[[...]]`, using its row numbers) or is a list of config dicts.
They are fitted exactly like sweep
trials and **added to the tuning data** (`TRIALS_CSV`, `source = "candidates"`), so the sweep tables
above include them; in W&B, each config's k=6 run is added to the k=6 sweep and its k=10 run to the k=10
sweep, tagged `candidate`. Configs already
run are skipped on re-running. The comparison table below shows just these configs. Put the chosen config into `ESTIMATORS` in `3_main_experiments.ipynb`.

In [ ]:
# in-degree -> existing W&B sweep: each config runs on these in-degrees, and its k=6 results go
# into the k=6 sweep, k=10 into the k=10 sweep (tagged "candidate")
CAND_SWEEPS = {6: "ey794fwc", 10: "p8lmbfxj"}

# Configs to run: rows of the top-config table above, picked by the row numbers it shows
# (or a list of config dicts written by hand). Names must be unique.
CANDIDATES = top_values.loc[[0, 1, 2, 3, 4, 13]]

# fitted exactly like sweep trials; rows are added to the tuning data (TRIALS_CSV, source="candidates",
# sweep_id of their in-degree's sweep). (config, in-degree) pairs already there are skipped on re-running.
cand = run_configs(
    CANDIDATES, k_values=list(CAND_SWEEPS), sweep_ids=CAND_SWEEPS, n_dags=SWEEP_DAGS, n_nodes=SWEEP_N,
    alpha=SWEEP_ALPHA, sample_sizes=SWEEP_SAMPLES, card=CARD, val_frac=VAL_FRAC,
    results_dir=RESULTS_DIR, n_cached=max(SAMPLE_SIZES), n_test=N_TEST,
    trial_csv=TRIALS_CSV, dag_offset=TUNE_DAG_OFFSET,
    wandb_project="neural-cpds-hparams",      # same project as the sweeps; None to skip W&B
    n_jobs=N_JOBS, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

configs:   0%|          | 0/6 [00:00<?, ?it/s]

mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,0.58179
mean_val_loss,7.96924
sample_size,1400
std_kl,0.07353


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,1.84409
mean_val_loss,9.72992
sample_size,1400
std_kl,0.20391


mean_kl,█▁
mean_val_loss,█▁
sample_size,▁█
frac_at_ceiling,0
mean_kl,0.59156
mean_val_loss,7.93457
sample_size,1400
std_kl,0.06773


KeyboardInterrupt: 

### Candidate comparison
The configs run on every (in-degree, sample size) by the candidate cell above:
mean KL over the tuning DAGs and regret % against the best config in each pair.

In [ ]:
cand_table = plotting.candidate_table(plotting.load_trials(RESULTS_DIR))
if cand_table.empty:
    print("no candidate runs yet: run the candidate cell above")